# 03 streaming + stopping

目标：理解流式输出和停止条件。streaming 不减少总计算量，但能显著改善用户感知延迟。


## 运行环境准备

这个 notebook 默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，适合在魔搭 Notebook 里快速学习。

如果你想用更大的模型，可以把 `MODEL_ID` 改成 `Qwen/Qwen2.5-7B-Instruct`，然后重启内核重新运行。


In [1]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 66.5 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 161.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [gradio]10/11 [gradio]client]

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-20 23:05:34,368 - modelscope - INFO - Target directory already exists, skipping creation.


MODEL_ID = Qwen/Qwen2.5-0.5B-Instruct
MODEL_PATH = /mnt/workspace/.cache/modelscope/models/Qwen/Qwen2___5-0___5B-Instruct


## 1. 加载组件和自定义停止条件

`StoppingCriteria` 可以在生成循环里检查最新 token，满足条件就停止。


In [3]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
    StoppingCriteria,
    StoppingCriteriaList,
    TextStreamer,
)


class StopOnTokenIds(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = set(stop_token_ids)

    def __call__(self, input_ids, scores, **kwargs):
        if not self.stop_token_ids:
            return False
        return input_ids[0, -1].item() in self.stop_token_ids


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 521.76it/s]


## 2. 准备 prompt 和生成配置


In [4]:
messages = [
    {"role": "system", "content": "你是一个大模型部署工程师。"},
    {"role": "user", "content": "用要点解释：为什么 streaming 能改善体验？"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

generation_config = GenerationConfig(
    max_new_tokens=160,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.05,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)


## 3. 流式输出

`TextStreamer` 会在生成过程中逐步打印新增文本。


In [ ]:
streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

stopping_criteria = StoppingCriteriaList(
    [StopOnTokenIds([tokenizer.eos_token_id])]
)

model.generate(
    **inputs,
    generation_config=generation_config,
    streamer=streamer,
    stopping_criteria=stopping_criteria,
)


Streaming技术在提高用户体验方面有以下几个关键点：

1. **实时性与响应时间**：流式传输允许在用户设备上即时处理数据，从而减少延迟和丢包问题。这对于需要快速响应的场景，如实时游戏、实时视频会议等非常有益。

2. **负载均衡与容错能力**：通过将数据分散到多个流中，流式传输可以显著降低单点故障的风险，同时还能根据负载情况自动调整分发策略，确保服务的高可用性和可靠性。

3. **可扩展性**：流式传输技术允许在不牺牲用户体验的情况下，灵活地扩展或缩减计算资源，适用于需要根据用户需求动态调整服务规模的场景。

4. **高效数据处理与分析**：通过实时


tensor([[151644,   8948,    198,  56568, 101909,  26288, 104949, 102121, 105503,
           1773, 151645,    198, 151644,    872,    198,  11622, 108164, 104136,
           5122, 100678,  16842,   8908,  22597, 104009, 101904,  11319, 151645,
            198, 151644,  77091,    198,  76509,  99361,  18493, 100627, 112458,
          99522,  18830, 116420,  99936,  27442,  48443,     16,     13,   3070,
         105143,  33071,  57218, 102808,  20450,    334,   5122,  88653,  28330,
         107468, 102496,  18493,  20002, 101044,  17447, 103260,  54542,  20074,
           3837, 101982, 101940, 112881,  33108, 101527,  67279,  86119,   1773,
         113423,  85106, 101098, 102808,   9370, 102122,   3837,  29524, 105143,
          99329,   5373, 105143,  87140,  99903,  49567,  99491, 107091,   3407,
             17,     13,   3070, 118878, 107101,  57218,  36629,  28726,  99788,
            334,   5122,  67338,  44063,  20074, 105211,  26939, 101213,  88653,
          15946,   3837,  88